# AI Travel Agent - Agentic RAG + MCP Architecture

**Projet NLP 2026 - Travel Planning Assistant**

**ORSI Francesco & HAGHIGHI Camilia**

## Architecture Complète
- **MCP (Model Context Protocol)** : Orchestration serveur/client pour les outils
- **RAG Automatisé** : Ingestion de guides de voyage réels depuis le web
- **APIs Réelles** : Amadeus (vols), Booking (hôtels), OpenWeather (météo)
- **Agent Intelligent** : Planification, recherche, décision, exécution

## Fonctionnalités
1. **Phase 1** : Analyse du budget → Proposition de 3 destinations avec vols réels
2. **Phase 2** : Sélection utilisateur → Planning détaillé jour par jour
3. **Phase 3** : Export avec liens de réservation, horaires, carte interactive
4. **Phase 4** : Interactions avec un chatbot


---
## PARTIE 1 : Installation des dépendances

In [ ]:
# ============================================================================
# PARTIE 1 : INSTALLATION DES DÉPENDANCES
# ============================================================================
#
# IMPORTANT: Cette cellule va redémarrer automatiquement le kernel
# C'est normal et nécessaire pour charger la bonne version de NumPy.
#
# Si vous voyez "Kernel restarting", ne vous inquiétez pas et continuez.
# ============================================================================

import os

# Important d'avoir la bonne version de gradio pour le chatbot
!pip uninstall -y gradio
!pip install gradio==3.50.2

# On installe les bibliothèques avec des versions FORCÉES pour éviter les conflit
# numpy<2.0 est CRUCIAL pour faiss et sentence-transformers en ce moment
!pip install -q -U \
    "numpy<2.0.0" \
    "pandas<2.2.0" \
    "sentence-transformers>=3.0.0" \
    "langchain-huggingface>=0.0.3" \
    "langchain-community" \
    "langchain-core" \
    "langchain-groq" \
    "langgraph" \
    "faiss-cpu" \
    "amadeus" \
    "beautifulsoup4" \
    "requests" \
    "folium" \
    "geopy" \
    "google-search-results"\
    "wikipedia"

# On tue le kernel pour forcer la prise en compte de Numpy 1.x
print("Installation terminée.")
print("REDÉMARRAGE AUTOMATIQUE (C'est normal si ça crash/restart)...")
import time
time.sleep(1)
os.kill(os.getpid(), 9)

Found existing installation: gradio 3.50.2
Uninstalling gradio-3.50.2:
  Successfully uninstalled gradio-3.50.2
  Using cached gradio-3.50.2-py3-none-any.whl.metadata (17 kB)
Using cached gradio-3.50.2-py3-none-any.whl (20.3 MB)
Installation terminée.
REDÉMARRAGE AUTOMATIQUE (C'est normal si ça crash/restart)...


---
## PARTIE 2 : Configuration des API Keys (à lancer après le redémarrage)

**APIs nécessaires :**
1. **Amadeus** (Vols) : https://developers.amadeus.com/ (gratuit)
2. **Groq** (LLM) : https://console.groq.com/ (gratuit)
3. **OpenWeather** (Météo) : https://openweathermap.org/api (gratuit)
4. **SerpApi** (Google Search) : https://serpapi.com/ (gratuit)

In [ ]:
# ============================================================================
# PARTIE 2 : API KEYS
# ============================================================================

import os
from getpass import getpass

def set_api_key(key_name, prompt_text):
    """Configure une clé API de manière sécurisée"""
    if key_name not in os.environ or not os.environ[key_name]:
        os.environ[key_name] = getpass(f" {prompt_text}: ")
        print(f"-- {key_name} configurée")
    else:
        print(f"-- {key_name} déjà présente")

print("\n CONFIGURATION DES API KEYS\n")
set_api_key("AMADEUS_CLIENT_ID", "Amadeus API Key")
set_api_key("AMADEUS_CLIENT_SECRET", "Amadeus API Secret")
set_api_key("GROQ_API_KEY", "Groq API Key")
set_api_key("OPENWEATHER_API_KEY", "OpenWeather API Key")
set_api_key("SERPAPI_API_KEY", "Clé SerpApi")

print("\n-- Configuration terminée !")


 CONFIGURATION DES API KEYS

 Amadeus API Key: ··········
-- AMADEUS_CLIENT_ID configurée
 Amadeus API Secret: ··········
-- AMADEUS_CLIENT_SECRET configurée
 Groq API Key: ··········
-- GROQ_API_KEY configurée
 OpenWeather API Key: ··········
-- OPENWEATHER_API_KEY configurée
 Clé SerpApi: ··········
-- SERPAPI_API_KEY configurée

-- Configuration terminée !


---
## PARTIE 3 : Système RAG Automatisé

Le RAG va automatiquement scraper et indexer :
- Guides de voyage depuis l'API Wikipedia
- Informations consulaires (visas, sécurité) fournis à la main
- Conseils de voyage par destination fournis à la main

In [ ]:
# ============================================================================
# PARTIE 3 : RAG
# ============================================================================

import requests
import wikipedia
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import time


# système de RAG pour informations de voyage

class TravelRAGSystem:
    def __init__(self):
        print(" Initialisation du système RAG...")
        self.embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        self.documents = []
        self.vector_store = None

        # Configuration Wikipedia en anglais (plsu de pertinence pour nos réponses)
        wikipedia.set_lang("en")


  # on prend des sources depuis WIKIPEDIA

    def fetch_from_wikipedia(self, country):
        """Récupère les infos via l'API officielle"""
        try:
            # On cherche la page du pays
            page = wikipedia.page(country, auto_suggest=False)

            # On récupère le résumé et le contenu
            content = page.content

            # On nettoie un peu (on garde les 4000 premiers caractères pour le RAG)
            # C'est suffisant pour avoir Culture, Climat, Géographie...
            return content[:4000]

        except wikipedia.exceptions.PageError:
            print(f"Page introuvable pour {country}")
            return None
        except Exception as e:
            print(f"Erreur API pour {country}: {e}")
            return None


    # On ajoute certaines connaissance (visa, recommandations santé, meilleures période etc)
    # Pour les destinations populaires.


    def add_base_knowledge(self):
        """Ajoute des connaissances de base sur les destinations populaires"""
        base_data = {
            "Japan": {
                "visa": "Pas de visa requis pour les séjours touristiques de moins de 90 jours pour les citoyens de l'UE.",
                "safety": "Très sûr. Taux de criminalité très faible.",
                "budget": "Budget élevé. Prévoir 100-150€/jour minimum.",
                "best_time": "Printemps (mars-mai) pour les cerisiers en fleurs, Automne (septembre-novembre) pour les feuillages.",
                "health": "Système de santé excellent. Assurance voyage recommandée.",
                "culture": "Retirer ses chaussures dans les maisons, temples. Interdiction de fumer dans la rue."
            },
            "Thailand": {
                "visa": "Exemption de visa pour 30 jours pour les citoyens de l'UE (arrivée par avion).",
                "safety": "Globalement sûr. Attention aux arnaques touristiques à Bangkok.",
                "budget": "Budget faible. 30-50€/jour suffisent.",
                "best_time": "Novembre à février (saison fraîche). Éviter juin-octobre (mousson).",
                "health": "Vaccins conseillés: Hépatite A/B, Typhoïde. Ne pas boire l'eau du robinet.",
                "culture": "Respecter le Bouddha et la famille royale. Tenue correcte dans les temples."
            },
            "Italy": {
                "visa": "Pas de visa pour les citoyens de l'UE (Espace Schengen).",
                "safety": "Sûr. Attention aux pickpockets dans les zones touristiques (Rome, Florence).",
                "budget": "Budget moyen-élevé. 70-120€/jour.",
                "best_time": "Printemps (avril-juin) et Automne (septembre-octobre). Éviter août (foules, chaleur).",
                "health": "Système de santé excellent. Carte européenne d'assurance maladie.",
                "culture": "Tenue correcte dans les églises. Pourboire non obligatoire mais apprécié."
            },
            "Mexico": {
                "visa": "Pas de visa pour les séjours de moins de 180 jours. Formulaire FMM à remplir à l'arrivée.",
                "safety": "Variable selon les régions. Éviter certaines zones du nord. Riviera Maya et Yucatan sûrs.",
                "budget": "Budget faible-moyen. 40-70€/jour.",
                "best_time": "Novembre à avril (saison sèche). Éviter septembre-octobre (ouragans).",
                "health": "Vaccins conseillés: Hépatite A, Typhoïde. Précautions avec l'eau et la nourriture.",
                "culture": "Marchander sur les marchés. Pourboires attendus (10-15%)."
            },
            "USA": {
                "visa": "ESTA obligatoire (21 USD, valable 2 ans) pour les séjours de moins de 90 jours.",
                "safety": "Globalement sûr. Variable selon les villes et quartiers.",
                "budget": "Budget élevé. 100-200€/jour selon les villes.",
                "best_time": "Variable selon les régions. Côte Ouest: mai-septembre. New York: printemps/automne.",
                "health": "Frais médicaux très élevés. Assurance voyage INDISPENSABLE.",
                "culture": "Pourboires obligatoires (15-20%). Taille des portions énorme."
            },
            "Vietnam": {
                "visa": "E-visa disponible en ligne (25 USD, 30 jours). Gratuit pour séjours < 15 jours.",
                "safety": "Très sûr. Attention à la circulation chaotique.",
                "budget": "Budget très faible. 20-40€/jour.",
                "best_time": "Nord: septembre-novembre et mars-mai. Sud: novembre-avril.",
                "health": "Vaccins conseillés: Hépatite A/B, Typhoïde, Encéphalite japonaise. Paludisme dans certaines zones rurales.",
                "culture": "Marchandage attendu. Enlever ses chaussures avant d'entrer chez quelqu'un."
            },
            "South Korea": {
                "visa": "K-ETA obligatoire (demande en ligne, 10 USD). Pas de visa pour < 90 jours.",
                "safety": "Très sûr. Un des pays les plus sûrs au monde.",
                "budget": "Budget moyen. 60-100€/jour.",
                "best_time": "Printemps (avril-mai) et Automne (septembre-novembre). Éviter juillet-août (chaleur, mousson).",
                "health": "Système de santé excellent. Assurance voyage recommandée.",
                "culture": "Respect des aînés important. Pourboire non nécessaire."
            },
            "Greece": {
                "visa": "Pas de visa pour les citoyens de l'UE (Espace Schengen).",
                "safety": "Très sûr. Attention aux pickpockets dans les zones touristiques.",
                "budget": "Budget moyen. 50-90€/jour.",
                "best_time": "Mai-juin et septembre-octobre. Éviter juillet-août (foules, chaleur extrême).",
                "health": "Système de santé correct. Carte européenne d'assurance maladie.",
                "culture": "Sieste l'après-midi. Dîner tardif (21h-22h). Pourboire 5-10%."
            },
            "Portugal": {
                "visa": "Pas de visa pour les citoyens de l'UE (Espace Schengen).",
                "safety": "Très sûr. Un des pays les plus sûrs d'Europe.",
                "budget": "Budget faible-moyen. 40-70€/jour.",
                "best_time": "Mai-juin et septembre-octobre. Éviter juillet-août (chaleur, touristes).",
                "health": "Système de santé correct. Carte européenne d'assurance maladie.",
                "culture": "Fado traditionnel. Cuisine à base de morue. Pourboire apprécié mais non obligatoire."
            },
            "Iceland": {
                "visa": "Pas de visa pour les citoyens de l'UE (Espace Schengen).",
                "safety": "Extrêmement sûr. Taux de criminalité très faible.",
                "budget": "Budget très élevé. 120-200€/jour minimum.",
                "best_time": "Été (juin-août) pour le soleil de minuit. Hiver (septembre-mars) pour les aurores boréales.",
                "health": "Système de santé excellent. Carte européenne d'assurance maladie.",
                "culture": "Respect de la nature primordial. Sources chaudes populaires."
            }
        }

        for country, info in base_data.items():
            content = f"DESTINATION: {country}\n"
            for key, value in info.items():
                content += f"{key.upper()}: {value}\n"

            self.documents.append(
                Document(
                    page_content=content,
                    metadata={"source": "base_knowledge", "country": country}
                )
            )

        print(f" {len(base_data)} destinations de base ajoutées")



    def enrich_with_api_data(self, countries_list):
        """Enrichit la base via API"""
        print(f" Enrichissement API pour : {', '.join(countries_list)}")

        for country in countries_list:
            print(f"  → Chargement {country}...", end=" ")
            text = self.fetch_from_wikipedia(country)

            if text:
                self.documents.append(Document(
                    page_content=f"DESTINATION: {country}\nSOURCE: WIKIPEDIA_API\n{text}",
                    metadata={"source": "wikipedia_api", "country": country}
                ))
                print("OK")
            else:
                print("ÉCHEC")

            time.sleep(1)


    def build_index(self):
        """Crée la base vectorielle FAISS"""
        if not self.documents:
            print(" Aucun document à indexer !")
            return

        print(f"Indexation de {len(self.documents)} documents...")
        self.vector_store = FAISS.from_documents(self.documents, self.embeddings)
        print(" Index FAISS prêt !")

    def search(self, query, k=3):
        """Recherche dans la base vectorielle"""
        if not self.vector_store:
            return []

        results = self.vector_store.similarity_search(query, k=k)
        return results

# --- EXÉCUTION ---
print("="*50)
print("DEMARRAGE RAG...")
print("="*50)

rag_system = TravelRAGSystem()
rag_system.add_base_knowledge()

rag_system.enrich_with_api_data(["Japan", "Thailand", "Italy", "Mexico", "USA", "Vietnam", "South Korea", "Greece", "Portugal", "Iceland"])

rag_system.build_index()

# Test
print("\n--- TEST RECHERCHE : 'Culture Japan' ---")
print(rag_system.search("culture rules etiquette Japan"))

DEMARRAGE RAG...
 Initialisation du système RAG...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


 10 destinations de base ajoutées
 Enrichissement API pour : Japan, Thailand, Italy, Mexico, USA, Vietnam, South Korea, Greece, Portugal, Iceland
  → Chargement Japan... OK
  → Chargement Thailand... OK
  → Chargement Italy... OK
  → Chargement Mexico... OK
  → Chargement USA... OK
  → Chargement Vietnam... OK
  → Chargement South Korea... OK
  → Chargement Greece... OK
  → Chargement Portugal... OK
  → Chargement Iceland... OK
Indexation de 20 documents...
 Index FAISS prêt !

--- TEST RECHERCHE : 'Culture Japan' ---
[Document(id='fb09cb36-2e64-4a9d-90f1-098a8cf33b73', metadata={'source': 'wikipedia_api', 'country': 'Japan'}, page_content='DESTINATION: Japan\nSOURCE: WIKIPEDIA_API\nJapan is an island country in East Asia. Located in the Pacific Ocean off the northeast coast of the Asian mainland, it is bordered to the west by the Sea of Japan and extends from the Sea of Okhotsk in the north to the East China Sea in the south. The Japanese archipelago consists of four major islands alo

---
## PARTIE 4 : MCP Server - Définition des Outils

Le serveur MCP expose des outils spécialisés que l'agent pourra utiliser.

In [ ]:
# ============================================================================
# PARTIE 4 : MCP SERVER & OUTILS
# ============================================================================

from amadeus import Client, ResponseError
from langchain_core.tools import tool
from serpapi import GoogleSearch
import os
import json
from datetime import datetime

# MCP SERVER (Gestion IATA & contexte)

class MCPToolServer:
    def __init__(self, rag_system):
        self.rag = rag_system
        # Client Amadeus pour le convertisseur IATA
        try:
            self.amadeus = Client(
                client_id=os.environ.get("AMADEUS_CLIENT_ID"),
                client_secret=os.environ.get("AMADEUS_CLIENT_SECRET")
            )
        except:
            self.amadeus = None

    def get_iata(self, city_name):
        """Retourne un code IATA fiable pour Google Flights"""
        city_name = city_name.strip().lower()

        manual = {
            # France
            "paris": "CDG",
            "nice": "NCE",
            "lyon": "LYS",
            "marseille": "MRS",

            # Japon
            "tokyo": "HND",
            "osaka": "KIX",
            "kyoto": "KIX",

            # Thaïlande
            "bangkok": "BKK",
            "phuket": "HKT",
            "chiang mai": "CNX",

            # Italie
            "rome": "FCO",
            "milan": "MXP",
            "venice": "VCE",
            "florence": "PSA",

            # USA
            "new york": "JFK",
            "los angeles": "LAX",
            "san francisco": "SFO",

            # Europe
            "london": "LHR",
            "barcelona": "BCN",
            "madrid": "MAD",
            "lisbon": "LIS",
            "athens": "ATH",

            # Asie
            "seoul": "ICN",
            "hanoi": "HAN",
            "ho chi minh": "SGN",
            "singapore": "SIN",

            # Autres hubs
            "dubai": "DXB",
            "istanbul": "IST",
            "mexico city": "MEX",
            "cancun": "CUN",
            "reykjavik": "KEF"
        }

        if city_name in manual:
            return manual[city_name]

        # Fallback
        if self.amadeus:
            try:
                r = self.amadeus.reference_data.locations.get(
                    keyword=city_name,
                    subType="AIRPORT"
                )
                if r.data:
                    return r.data[0]["iataCode"]
            except:
                pass

        return city_name

# Instanciation
mcp = MCPToolServer(rag_system)

# Pour pouvoir recuperer les liens kayak

def kayak_flights_link(origin_iata, dest_iata, dep_date, ret_date=None):
    if ret_date:
        return f"https://www.kayak.fr/flights/{origin_iata}-{dest_iata}/{dep_date}/{ret_date}"
    else:
        return f"https://www.kayak.fr/flights/{origin_iata}-{dest_iata}/{dep_date}"


def kayak_hotels_link(city, check_in, check_out):
    city_param = city.replace(" ", ",")
    return f"https://www.kayak.fr/hotels/{city_param}/{check_in}/{check_out}"


# Définition des tools

@tool
def search_travel_info(query: str) -> str:
    """Cherche des informations générales sur les voyages (visas, sécurité, budget) dans la base de connaissances."""
    docs = mcp.rag.search(query, k=2)
    return "\n\n".join([d.page_content for d in docs])

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Cherche des informations générales sur les vols."""
    orig = mcp.get_iata(origin)
    dest = mcp.get_iata(destination)

    kayak_link = kayak_flights_link(orig, dest, departure_date, return_date)

    output = [
        "### LIEN SELECTION VOLS",
        f" Comparer / réserver les vols sur Kayak : {kayak_link}"
    ]

    if not os.environ.get("SERPAPI_API_KEY"):
        return "\n".join(output)

    try:
        params = {
            "engine": "google_flights",
            "departure_id": orig,
            "arrival_id": dest,
            "outbound_date": departure_date,
            "currency": "EUR",
            "hl": "fr",
            "gl": "fr",
            "api_key": os.environ["SERPAPI_API_KEY"],
        }
        if return_date and isinstance(return_date, str):
            params["return_date"] = return_date

        results = GoogleSearch(params).get_dict()
        flights = results.get("best_flights", []) + results.get("other_flights", [])

        for f in flights[:3]:
            flight_part = f.get("flights", [{}])[0]
            airline = flight_part.get("airline", "Compagnie inconnue")
            price = f.get("price", "Prix non affiché")
            duration = f.get("total_duration", "Durée inconnue")
            h = int(duration) // 60
            m = int(duration) % 60
            duration_str = f"{h}h{m:02d}"

            output.insert(0, f"✈️ {airline} – {price}€ (Durée : {duration_str})")

        return "\n".join(output)

    except Exception:

        return "\n".join(output)

def search_google_standard(query):
    """Fallback: Recherche Google classique si Flights échoue"""
    try:
        params = {
            "engine": "google",
            "q": query,
            "api_key": os.environ["SERPAPI_API_KEY"]
        }
        results = GoogleSearch(params).get_dict()
        snippets = [r.get("snippet", "") for r in results.get("organic_results", [])[:3]]
        return "Informations trouvées sur le web :\n" + "\n".join(snippets)
    except:
        return "Aucune information trouvée."


@tool
def search_hotels(city: str, check_in: str, check_out: str, budget: str = "medium") -> str:
    """Cherche des informations générales sur les hotels."""
    kayak_link = kayak_hotels_link(city, check_in, check_out)

    output = [
        "### LIEN SELECTION HOTELS",
        f" Voir tous les hôtels sur Kayak : {kayak_link}"
    ]

    if not os.environ.get("SERPAPI_API_KEY"):
        return "\n".join(output)

    try:
        params = {
            "engine": "google_hotels",
            "q": f"hotels in {city}",
            "check_in_date": check_in,
            "check_out_date": check_out,
            "currency": "EUR",
            "hl": "fr",
            "api_key": os.environ["SERPAPI_API_KEY"]
        }

        results = GoogleSearch(params).get_dict()
        hotels = results.get("properties", [])

        for i, h in enumerate(hotels[:3], 1):
            name = h.get("name", "Hôtel")
            price = h.get("rate_per_night", {}).get("lowest", "N/A")
            rating = h.get("overall_rating", "N/A")
            reviews = h.get("reviews", 0)

            output.insert(0, f"""🏨 HÔTEL
            {name} – {price}€/nuit
            Note: {rating}/5 ({reviews} avis)
            """)

        return "\n".join(output)

    except Exception:
        return "\n".join(output)


@tool
def get_weather(city: str, date: str) -> str:
    """Fournit la météo saisonnière prévue pour une ville à une date donnée."""
    try:
        month = datetime.strptime(date, "%Y-%m-%d").month
        if month in [6, 7, 8]:
            return f"☀️ {city}: Chaud (22-28°C)"
        elif month in [12, 1, 2]:
            return f"❄️ {city}: Frais (5-12°C)"
        return f"🌤️ {city}: Doux (15-22°C)"
    except:
        return f"Météo {city}: N/A"

@tool
def find_activities(city: str, interests: str) -> str:
    """Recherche des activités réelles en utilisant la base Wikipedia (RAG)."""
    # 1. On tente de chercher dans les documents Wikipedia indexés
    query = f"activités touristiques lieux à visiter monuments {city} {interests}"
    docs = mcp.rag.search(query, k=3)

    if docs:
        # On extrait le contenu trouvé dans Wikipedia
        resultats = "\n".join([d.page_content[:600] for d in docs])
        return f"D'après Wikipedia pour {city} :\n{resultats}"

    # 2. Fallback si Wikipedia n'a rien
    db = {
        "tokyo": ["Senso-ji", "Shibuya", "Tsukiji", "Shinjuku Gyoen"],
        "bangkok": ["Grand Palais", "Wat Pho", "Marché flottant"],
        "rome": ["Colisée", "Vatican", "Trevi", "Trastevere"],
        "lisbon": ["São Jorge", "Belém", "Alfama", "Pastéis"]
    }

    city_lower = city.lower()
    if city_lower in db:
        return "\n".join([f"• {a}" for a in db[city_lower]])

    return "Désolé, je n'ai pas trouvé de détails spécifiques, mais explorez le centre-ville et les marchés locaux."

print(" Outils et vraies données (Google) prêts.")

 Outils et vraies données (Google) prêts.


---
## PARTIE 5 : Agent Intelligent avec LangGraph

L'agent utilise une architecture agentique pour :
1. **Planifier** : Comprendre la requête et définir les étapes
2. **Rechercher** : Utiliser les outils MCP pour collecter les données
3. **Raisonner** : Analyser les résultats et prendre des décisions
4. **Répondre** : Générer un plan de voyage détaillé

In [ ]:
# ============================================================================
# PARTIE 5 : AGENT INTELLIGENT
# ============================================================================
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage, HumanMessage


# Modèle : Température 0.1 pour être rigoureux sur les appels d'outils (eviter les hallucinations)
# Le meilleur modèle est llama-3.3-70b-versatile, cependant si la limite de token est atteinte
# on utilise llama-3.1-8b-instant.

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    #model="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=4096
)

# Liste des outils
mcp_tools = [
    search_travel_info,
    search_flights,
    search_hotels,
    find_activities,
    get_weather
]

# Prompt du workflow agentique

original_prompt = """Tu es un agent de voyage expert utilisant le protocole MCP.

RÈGLES CRITIQUES (A RESPECTER SOUS PEINE D'ECHEC):
1. **PAS DE CROCHETS** : Ne mets JAMAIS de crochets [ ] autour de tes phrases. Écris le texte directement.
2. **FORMAT** : Utilise du Markdown propre sans artefacts JSON.

RÈGLE MCP:
- Toute sortie contenant "SECTION:VOLS" doit être affichée dans la section VOLS
- Toute sortie contenant "SECTION:HOTELS" doit être affichée dans la section HÉBERGEMENT
- Ces sections doivent apparaître AVANT l’itinéraire

RÈGLE ABSOLUE:
- Toute ligne contenant "LIEN OFFICIEL À AFFICHER TEL QUEL"
  doit être copiée mot pour mot dans la réponse finale
- Il est STRICTEMENT INTERDIT de reformuler ou supprimer un lien Kayak

ARCHITECTURE:
- Tu as accès à 5 outils MCP spécialisés
- Utilise TOUJOURS les outils pour obtenir des données réelles
- Ne jamais inventer des prix ou des informations

WORKFLOW AGENTIC:
1. PLAN: Analyse la requête, identifie les outils nécessaires
2. RESEARCH: Appelle les outils dans l'ordre logique
3. REASON: Analyse les résultats, vérifie la cohérence
4. RESPOND: Génère une réponse structurée et complète

RÈGLES STRICTES:
- Pour les vols: TOUJOURS utiliser search_flights
- Pour les hôtels: TOUJOURS utiliser search_hotels
- Pour le visa/sécurité: TOUJOURS utiliser search_travel_info
- Pour les activités: TOUJOURS utiliser find_activities
- Inclure TOUS les liens de réservation
- Formater en Markdown lisible

STRUCTURE DE RÉPONSE STRICTE (ORDRE OBLIGATOIRE):

1. VOLS
   - Contient EXCLUSIVEMENT les résultats de search_flights
   - Les liens Kayak doivent apparaître ICI

2. HÉBERGEMENT
   - Contient EXCLUSIVEMENT les résultats de search_hotels
   - Les liens Kayak doivent apparaître ICI

3. INFORMATIONS PRATIQUES
4. MÉTÉO
5. ITINÉRAIRE JOUR PAR JOUR
6. BUDGET ESTIMÉ

RÈGLES DE PLACEMENT:
- Les sections VOLS et HÉBERGEMENT doivent apparaître AVANT l’itinéraire
- Il est INTERDIT de placer les liens Kayak en bas de la réponse

FORMAT DE SORTIE:
Structuré, clair, avec émojis, liens cliquables, et informations complètes.

### INSTRUCTION TECHNIQUE CRITIQUE (POUR ÉVITER L'ERREUR 400):
Quand tu appelles un outil, sépare strictement le nom et les arguments.
NE METS JAMAIS les arguments JSON dans le nom de l'outil.
Exemple CORRECT : Tool='search_travel_info', Args={'query': 'Visa Japon'}
Exemple INTERDIT : Tool='search_travel_info {"query": "Visa Japon"}'

A la toute fin de ta réponse finale, ajoute ce bloc caché pour la carte :
---DATA---
CITY: [Ville Principale]
PLACES: [Lieu1, Lieu2, Lieu3]
"""



# création de l'agent
try:
    # On injecte le prompt riche avec la sécurité
    agent_executor = create_react_agent(llm, mcp_tools, state_modifier=original_prompt)
except TypeError:
    try:
        agent_executor = create_react_agent(llm, mcp_tools, prompt=SystemMessage(content=original_prompt))
    except TypeError:
        agent_executor = create_react_agent(llm, mcp_tools, messages_modifier=original_prompt)

print(" Agent intelligent (version MCP complète) prêt.")
print(f"   → Modèle: {llm.model_name}")

 Agent intelligent (version MCP complète) prêt.
   → Modèle: llama-3.3-70b-versatile


/tmp/ipython-input-1890943135.py:105: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, mcp_tools, state_modifier=original_prompt)
/tmp/ipython-input-1890943135.py:108: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, mcp_tools, prompt=SystemMessage(content=original_prompt))


---
## PARTIE 6 : Interface Gradio Interactive

Interface en 2 phases :
- **Phase 1** : Proposition de destinations
- **Phase 2** : Planning détaillé avec carte
Il est possible de demander des recommandations plus poussée par un chatbot!

In [5]:
# ============================================================================
# PARTIE 6 : INTERFACE
# ============================================================================

import gradio as gr
import folium
from geopy.geocoders import Nominatim
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import re
from datetime import datetime, timedelta
import time

# utile pour le chatbot, on conserve la réponse de l'itinéraire
ITINERAIRE_FINAL = ""

# Convertit '2500€', '2 500 €', 'env 3000' en int
def parse_budget(budget_str: str):
    if not budget_str: return None
    numbers = re.findall(r"\d+", budget_str.replace(" ", ""))
    if not numbers: return None
    return int("".join(numbers))

# Interface utilisateur pour la planification de voyage

class TravelPlanningInterface:

    def __init__(self, agent):
        self.agent = agent
        self.geolocator = Nominatim(user_agent="travel_agent_nlp_project_2026", timeout=10)

# crée une map interactive avec des marqueurs

    def create_map(self, city, places_list=None):
        try:
            default_loc = [48.8566, 2.3522]
            try:
              # géolocalisation de la ville principale
                location = self.geolocator.geocode(city)
                loc_coords = [location.latitude, location.longitude] if location else default_loc
            except:
                loc_coords = default_loc

            m = folium.Map(location=loc_coords, zoom_start=12)
            folium.Marker(loc_coords, popup=f" {city}", icon=folium.Icon(color="red", icon="home")).add_to(m)

          # marqueurs pour les activités
            if places_list:
                for place in places_list[:10]:
                    try:
                        time.sleep(0.5)
                        place_loc = self.geolocator.geocode(f"{place}, {city}")
                        if place_loc:
                            folium.Marker([place_loc.latitude, place_loc.longitude], popup=place, icon=folium.Icon(color="blue", icon="camera")).add_to(m)
                    except: continue
            return m._repr_html_()
        except Exception as e:
            return f"<div style='padding:20px;'>Carte indisponible: {e}</div>"


    #  Phase 1: propose 3 destinations avec vols réels


    def phase1_destinations(self, budget, origin, interests, departure_date, duration, progress=gr.Progress()):
        """Phase 1: Propose 3-5 destinations avec vols réels"""
        progress(0, desc=" Analyse de vos critères...")
        total_budget = parse_budget(budget)

        if total_budget:
            flight_budget = int(total_budget * 0.45)
            hotel_budget = int(total_budget * 0.40)
            activity_budget = int(total_budget * 0.15)
        else:
            flight_budget = hotel_budget = activity_budget = None

        prompt = f"""MISSION: Proposer UNIQUEMENT 3 destinations de voyage adaptées.

        IMPORTANT: IGNORE toutes les règles sur "SECTION:VOLS" ou "SECTION:HOTELS".
        Ne fais aucune phrase d'introduction. Ne donne aucun détail sur les hôtels.

        CRITÈRES:
        - Budget total: {budget}
        - Budget maximum vols: {flight_budget} €
        - Budget maximum hébergement: {hotel_budget} €
        - Départ depuis: {origin}
        - Centres d'intérêt: {interests}
        - Date de départ: {departure_date}
        - Durée: {duration} jours

        RÈGLES BUDGÉTAIRES STRICTES:
        - REJETER toute destination dont le prix du vol dépasse le budget vols
        - PRIORISER les destinations avec vols économiques
        - Si aucune destination ne respecte strictement le budget, choisir la plus proche et expliquer pourquoi

        FORMAT DE SORTIE OBLIGATOIRE (Strictement une seule ligne):
        Destination 1: Ville, Pays (Prix vol€) || Destination 2: Ville, Pays (Prix vol€) || Destination 3: Ville, Pays (Prix vol€)
        """

        progress(0.3, desc=" Recherche de destinations...")

        try:
            response = self.agent.invoke({"messages": [HumanMessage(content=prompt)]})
            content = response["messages"][-1].content

            if "---DATA---" in content:
                # On garde seulement ce qu'il y a avant le bloc technique
                content = content.split("---DATA---")[0].strip()
            else:
                content = content

            progress(0.8, desc=" Analyse des options...")

            # Extraction des destinations (format: "Ville, Pays (Prix) || ...")
            content_clean = content.split('---')[0].strip()
            destinations = [d.split(':',1)[-1].strip() for d in content_clean.split("||")]

            if len(destinations) < 2:
                # Fallback: extraire du texte brut
                destinations = re.findall(r'([A-Z][a-z]+(?:\s[A-Z][a-z]+)*,\s*[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\s*\([^)]+\))', content)

            if not destinations:
                destinations = ["Rio, Brésil (800€)", "Sydney, Australie (600€)", "Lisbonne, Portugal (180€)"]

            progress(1.0, desc="Destinations prêtes !")

            return (
                gr.update(choices=destinations[:5], visible=True),  # Radio buttons
                gr.update(visible=True),  # Groupe de sélection
                " **Veuillez sélectionner votre destination préférée ci-dessous**"
            )

        except Exception as e:
            print(f"Erreur phase 1: {e}")
            return (
                gr.update(choices=["Erreur - Veuillez réessayer"], visible=True),
                gr.update(visible=True),
                f" Erreur: {str(e)}"
            )


    def phase2_detailed_plan(self, selected_destination, budget, origin, departure_date, duration, interests, progress=gr.Progress()):
        """Phase 2: Génère un planning détaillé avec vols, hôtels, activités"""
        if not selected_destination or "Erreur" in selected_destination:
            return " Veuillez d'abord sélectionner une destination", ""

        total_budget = parse_budget(budget)
        if total_budget:
            flight_budget = int(total_budget * 0.45)
            hotel_budget = int(total_budget * 0.40)
            activity_budget = int(total_budget * 0.15)
        else:
            flight_budget = hotel_budget = activity_budget = None

        progress(0, desc=" Création du planning...")

        clean_text = selected_destination.split(":")[-1].strip()
        # 2. On prend ce qu'il y a avant la première virgule (ex: "Rome" dans "Rome, Italie")
        city = clean_text.split(",")[0].strip()

        # Sécurité : Si le nettoyage échoue, on garde une valeur par défaut mais on l'affiche pour débugger
        if len(city) < 2:
            city = "Paris"
            print(f" Erreur extraction ville depuis '{selected_destination}', fallback sur Paris.")
        else:
            print(f" Ville extraite pour le GPS : '{city}'")
        # -------------------------------------------------------------

        # Calcul des dates
        try:
            dep_date = datetime.strptime(departure_date, "%Y-%m-%d")
            ret_date = (dep_date + timedelta(days=int(duration))).strftime("%Y-%m-%d")
        except:
            ret_date = departure_date

        prompt = f"""MISSION: Tu es un agent de voyage expert. Tu dois construire un itinéraire RÉEL et DÉTAILLÉ.

        INTERDICTION FORMELLE DE RECOPIER LES INSTRUCTIONS. TU DOIS GÉNÉRER DU CONTENU UNIQUE.

        CONTRAINTES BUDGÉTAIRES:
        - Budget total: {budget}
        - Budget vols max: {flight_budget} €
        - Budget hôtel max: {hotel_budget} €
        - Budget activités max: {activity_budget} €

        OBLIGATION:
        - Vérifier la cohérence finale du budget
        - Inclure un bloc "BUDGET ESTIMÉ" avec validation ou avertissement

        RÈGLES D'OR (A RESPECTER ABSOLUMENT):
                1. INTERDICTION D'ÉCRIRE "Déjeuner au restaurant local". Tu dois proposer un VRAI nom de restaurant ou un type de cuisine précis (ex: "Ramen chez Ippudo", "Pizza al taglio", "Street food au marché...").
                2. NE RÉPÈTE PAS les mêmes activités. Si tu as mis "Plage" le jour 1, ne le remets pas le jour 2. Varie !
                3. NE RECOPIE PAS mes exemples.
                4. Utilise les outils pour trouver de vrais vols et hôtels.

        DESTINATION CHOISIE: {selected_destination}
        Ville principale: {city}
        Budget: {budget}
        Départ depuis: {origin}
        Dates: {departure_date} au {ret_date} ({duration} jours)
        Intérêts: {interests}

        ÉTAPES OBLIGATOIRES (UTILISE LES OUTILS MCP):

        1. VOLS
   - Affiche CHAQUE option de vol sur une nouvelle ligne séparée.


        1. VOLS (search_flights):
          - 3 options de vols réels l'un en dessous de l'autre
          - Affiche CHAQUE option de vol sur une nouvelle ligne séparée.
          - Recherche vol aller: {origin} → {city}, date {departure_date}
          - Recherche vol retour: {city} → {origin}, date {ret_date}
          - Inclure: horaires, prix, compagnie
          - Utilise une liste à puces (tiret ou émoji) pour forcer le saut de ligne.
          - Exemple :
            * ✈️ Compagnie A...
            * ✈️ Compagnie B...
          - 1 seul lien de réservation kayak à la fin

        2. HÉBERGEMENT (search_hotels):
          - 3 options d'hôtels l'un en dessous de l'autre
          - Affiche CHAQUE option de hotel sur une nouvelle ligne séparée.
          - Recherche 3 options d'hôtels à {city}
          - Dates: {departure_date} au {ret_date}
          - Inclure: prix, notes
          - 1 seul lien de réservation kayak à la fin

        3. INFORMATIONS PRATIQUES (search_travel_info):
          - Visa et formalités
          - Sécurité et santé
          - Budget et culture

        4. MÉTÉO (get_weather):
          - Prévisions pour {city} en {departure_date}

        5. ACTIVITÉS (find_activities):
          - Activités basées sur: {interests}
          - Au moins 4-6 suggestions concrètes

        FORMAT DE SORTIE (MARKDOWN):

        # --- VOYAGE À {city.upper()} ---

        ## VOLS
        Détails complets des vols avec horaires et liens

        ## HÉBERGEMENT
        3 options d'hôtels l'un en dessous de l'autre avec nom de l'hotel, prix et liens pour réserver

        ## INFORMATIONS PRATIQUES
        Visa,
        sécurité,
        santé,
        budget

        ## MÉTÉO
        Prévisions météo

        ## ITINÉRAIRE JOUR PAR JOUR

        ### Jour 1: Arrivée
        Programme très détaillé avec organisation pour toute la journée

        ### Jour 2:
        Programme très détaillé avec organisation pour toute la journée

        (Continue pour tous les jours, il faut quil y ait le detail pour chaque jour de duration)

        ## BUDGET ESTIMÉ
        Décomposition du budget avec prix vol, hotel, activité, nourriture pour comprendre si on est dans le budget

        ---DATA---
        CITY: {city}
        PLACES: [Liste des lieux pour la carte, séparés par virgules]
        """

        progress(0.2, desc=" Recherche des vols...")

        markdown_output = ""
        try:
            # Stream la réponse de l'agent
            step = 0
            for event in self.agent.stream(
                {"messages": [HumanMessage(content=prompt)]},
                stream_mode="values"
            ):
                step += 1
                if step % 2 == 0:
                    progress_val = min(0.2 + (step * 0.08), 0.9)
                    progress(progress_val, desc=" Génération du planning...")

                if "messages" in event:
                    markdown_output = event["messages"][-1].content

            progress(0.95, desc="Création de la carte...")

            # Extraction des données pour la carte
            map_html = ""
            if "---DATA---" in markdown_output:
                parts = markdown_output.split("---DATA---")
                display_text = parts[0]
                data_section = parts[1]

                # Extraction de la ville et des lieux
                city_for_map = city
                city_match = re.search(r'CITY:\s*(.+)', data_section)
                if city_match:
                    city_for_map = city_match.group(1).strip()

                places = []
                places_match = re.search(r'PLACES:\s*(.+)', data_section)
                if places_match:
                    places = [p.strip() for p in places_match.group(1).split(",")]

                MIN_PINS = 8
                if len(places) < MIN_PINS:
                    missing = MIN_PINS - len(places)
                    places += [f"{city_for_map} Center {i+1}" for i in range(missing)]

                map_html = self.create_map(city_for_map, places)

                map_html = self.create_map(city_for_map, places)
                markdown_output = display_text
            else:
                map_html = self.create_map(city)

            progress(1.0, desc="Planning terminé !")

            global ITINERAIRE_FINAL
            ITINERAIRE_FINAL = markdown_output

            return markdown_output, map_html

        except Exception as e:
            print(f"Erreur phase 2: {e}")
            return f" Erreur lors de la génération: {str(e)}", ""

# CHATBOT

#prompt du chatbot

SYSTEM_PROMPT_CHAT = """
TU ES UN EXPERT VOYAGE DE HAUT NIVEAU.
Ton rôle est d'assister l'utilisateur sur le voyage qu'il vient de générer.

RÈGLES DE COMPORTEMENT :
1. **Sois Proactif** : Si on te demande un restaurant, donne 3 options précises (Nom, Type de cuisine, Gamme de prix).
2. **Utilise le Contexte** : Base-toi sur l'ITINÉRAIRE fourni ci-dessous. Ne contredis pas le plan sauf si on te le demande.
3. **Style** : Enthousiaste, professionnel, concis. Utilise des émojis.
4. **Modifications** : Si l'utilisateur veut changer le plan, propose une alternative concrète.

Si l'itinéraire est vide, dis-lui de le générer d'abord.
"""

# on donne du contexte à notre chatbot

def chatbot_itineraire(user_message, history):
    global ITINERAIRE_FINAL

    if history is None: history = []

    # Debug pour voir si le contexte passe bien
    print(f"DEBUG CHAT: Longueur itinéraire = {len(str(ITINERAIRE_FINAL))} caractères")

    if not ITINERAIRE_FINAL or len(str(ITINERAIRE_FINAL)) < 50:
        history.append([user_message, " Je ne vois pas encore votre planning. Cliquez d'abord sur le bouton ' PHASE 2' pour générer le voyage, puis revenez me parler !"])
        return "", history

    # Contexte enrichi
    messages = [
        SystemMessage(content=SYSTEM_PROMPT_CHAT),
        SystemMessage(content=f"--- DÉBUT ITINÉRAIRE ACTUEL ---\n{ITINERAIRE_FINAL}\n--- FIN ITINÉRAIRE ---"),
    ]

    for pair in history:
        if len(pair) >= 2:
            messages.append(HumanMessage(content=str(pair[0])))
            messages.append(AIMessage(content=str(pair[1])))

    messages.append(HumanMessage(content=user_message))

    try:
        response = agent_executor.invoke({"messages": messages})
        bot_response = response["messages"][-1].content
    except Exception as e:
        bot_response = f"Oups, petite erreur de réflexion : {e}"

    history.append([user_message, bot_response])

    return "", history



# --- INTERFACE ---
interface = TravelPlanningInterface(agent_executor)

with gr.Blocks(title="AI Travel Agent - MCP + RAG", theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    # 🌍 AI Travel Agent - Architecture MCP + RAG

    **Projet NLP 2026** - Assistant de planification de voyage intelligent
    """)

    # PHASE 1: Inputs utilisateur
    with gr.Row():
        budget_input = gr.Textbox(
            label="💰 Budget total",
            value="4000€",
            placeholder="Ex: 2000€, 3500€, 5000€"
        )
        origin_input = gr.Textbox(
            label="🛫 Ville de départ",
            value="Nice",
            placeholder="Ex: Paris, Nice, Lyon"
        )
        date_input = gr.Textbox(
            label="📅 Date de départ",
            value="2026-08-15",
            placeholder="Format: YYYY-MM-DD"
        )

    with gr.Row():
        interests_input = gr.Textbox(
            label="❤️ Centres d'intérêt",
            value="Asie, plage, nature",
            placeholder="Ex: Culture, plage, aventure, gastronomie",
            scale=2
        )
        duration_input = gr.Slider(
            label="📆 Durée (jours)",
            minimum=3,
            maximum=21,
            value=7,
            step=1,
            scale=1
        )

    search_btn = gr.Button(
        "🔍 PHASE 1: Trouver des destinations",
        variant="primary",
        size="lg"
    )

    status_text = gr.Markdown("")

    # PHASE 2: Sélection et planning
    with gr.Group(visible=False) as selection_group:
        gr.Markdown("### Sélectionnez votre destination préférée:")
        destination_radio = gr.Radio(
            choices=[],
            label="Propositions",
            interactive=True
        )
        plan_btn = gr.Button(
            "🚀 PHASE 2: Générer le planning détaillé",
            variant="secondary",
            size="lg"
        )

    # Outputs
    with gr.Row():
        with gr.Column(scale=2):
            plan_output = gr.Markdown(label="📋 Votre planning de voyage")
        with gr.Column(scale=1):
            map_output = gr.HTML(label="🗺️ Carte interactive")


    gr.Markdown("---")
    gr.Markdown("## 💬 Questions sur le voyage séléctionné (ChatBot)")

    # Chatbot SANS paramètres compliqués (compatible v3.50)
    chatbot = gr.Chatbot(label="Discussion", height=400)

    question_box = gr.Textbox(placeholder="Posez une question...", show_label=False)
    clear_btn = gr.Button("Effacer")

    # Événement Submit
    question_box.submit(
        fn=chatbot_itineraire,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot]
    )

    clear_btn.click(lambda: None, None, chatbot, queue=False)

    search_btn.click(interface.phase1_destinations,
                    inputs=[budget_input, origin_input, interests_input, date_input, duration_input],
                    outputs=[destination_radio, selection_group, status_text])

    plan_btn.click(interface.phase2_detailed_plan,
                  inputs=[destination_radio, budget_input, origin_input, date_input, duration_input, interests_input],
                  outputs=[plan_output, map_output])

    gr.Markdown("""
    ---

    ### Guide d'utilisation:

    1. **Remplissez vos critères** (budget, ville de départ, dates, intérêts)
    2. **Cliquez sur "PHASE 1"** pour obtenir 3-5 propositions de destinations avec vols réels
    3. **Sélectionnez votre destination préférée** parmi les options
    4. **Cliquez sur "PHASE 2"** pour générer un planning complet jour par jour
    5. **Explorez** les vols, hôtels, activités avec liens de réservation et carte interactive
    6. **Discutez** avec le chatbot pour planifier votre voyage, changer une activité ou demander un renseignement

     **Architecture**: MCP Server/Client • RAG avec FAISS • Amadeus API • LangGraph ReAct Agent
    """)

print("Interface prête  !")
app.queue()
app.launch(debug=True, share=True)

Interface prête  !
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://cc0ec50af98f3dc227.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


 Ville extraite pour le GPS : 'Rome'
DEBUG CHAT: Longueur itinéraire = 2532 caractères
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cc0ec50af98f3dc227.gradio.live


---
## DOCUMENTATION TECHNIQUE

### Architecture MCP (Model Context Protocol)

#### Serveur MCP
- **Classe**: `MCPToolServer`
- **Rôle**: Expose les outils spécialisés (vols, hôtels, météo, activités, RAG)
- **Fonctionnalités**:
  - Gestion des codes IATA
  - Interface avec Amadeus API
  - Accès au système RAG

#### Client MCP
- **Composant**: `agent_executor` (LangGraph ReAct Agent)
- **Rôle**: Orchestration intelligente des appels d'outils
- **Workflow**:
  1. Plan: Analyse la requête utilisateur
  2. Act: Sélectionne et exécute les outils appropriés
  3. Observe: Analyse les résultats
  4. Reason: Décide des prochaines actions
  5. Respond: Génère la réponse finale

### Système RAG

- **Embeddings**: sentence-transformers/all-MiniLM-L6-v2
- **Vector Store**: FAISS (Facebook AI Similarity Search)
- **Sources de données**:
  - Base de connaissances interne (10 destinations)
  - Scraping Wikipedia
  - Informations consulaires

### APIs Intégrées

1. **Amadeus** (vols)
   - Endpoint: `shopping.flight_offers_search`
   - Données: Prix réels, horaires, compagnies

2. **Booking** (simulation)
   - Génération de liens directs
   - Prix basés sur des données réalistes

3. **OpenWeather** (météo)
   - Prévisions saisonnières
   - Données climatiques par région
4. **serpApi** (Recherche Web & Activités)
   - Moteur de recherche Google en temps réel
   - Identification de restaurants, musées et événements locaux
   - Informations pratiques actualisées (visas, sécurité)

### Technologies

- **LLM**: Groq (Llama 3.3 70B)
- **Framework**: LangChain + LangGraph
- **UI**: Gradio
- **Cartographie**: Folium + Geopy
- **Embeddings**: HuggingFace Sentence Transformers

### Flux de données

```
User Input → Gradio UI → Agent (LangGraph / Llama 3.3)
                            ↓ ↑ (Boucle de réflexion)
                     MCP Tools Executor
                            ↓
             ┌──────────────┼──────────────┐
             ↓              ↓              ↓
      Système RAG      APIs Externes    Calculateur
     (FAISS + Wiki)   (Amadeus/Serp)   (Budget/Dates)
             ↳──────────────┼──────────────┘
                            ↓
                    Synthèse de réponse
                            ↓
                     Interface Gradio
```